# JAX Ant-Count Curriculum

Start from the `25x25`, 3-bit communication policy and progressively increase the ant count. The warm-start critic surgery and shared render/vault workflow live in `ant_byte_env.notebook_workflows`.


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
{"project_root": PROJECT_ROOT, **runtime_status}


In [ ]:
import importlib

import jax

from ant_byte_env import notebook_workflows as workflows
from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Curriculum Settings


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "communication_bits.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)

COMMUNICATION_BITS = 3
SOURCE_COMMUNICATION_CHECKPOINT = PROJECT_ROOT / "runs/notebooks/communication_bits_25x25/3_bits/checkpoints/model.pkl"
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "ant_count_25x25_3_bits"
MEDIA_DIR = RUN_DIR / "media"

TRAINING_ARGS = workflows.ant_count_training_args(
    experiment.args,
    communication_bits=COMMUNICATION_BITS,
)
SOURCE_NUM_ANTS = int(TRAINING_ARGS.get("num_ants", 1))
ANT_STAGES = [2, 3, 4, 6, 8]
GLOBAL_UPDATE_CAP = int(experiment.metadata.get("ant_global_update_cap", 2000))
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
NUM_ENVS = int(TRAINING_ARGS["num_envs"])
NUM_STEPS = int(TRAINING_ARGS["num_steps"])
UPDATE_TIMESTEPS = workflows.update_timesteps(num_envs=NUM_ENVS, num_steps=NUM_STEPS)
COMMON_ARGS = workflows.config_common_args(TRAINING_ARGS, exclude=workflows.ANT_COUNT_ARG_EXCLUDES)

workflows.validate_ant_count_stages(ant_stages=ANT_STAGES, source_num_ants=SOURCE_NUM_ANTS)
if not SOURCE_COMMUNICATION_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing source checkpoint: {SOURCE_COMMUNICATION_CHECKPOINT}")

{
    "source_checkpoint": SOURCE_COMMUNICATION_CHECKPOINT,
    "communication_bits": COMMUNICATION_BITS,
    "source_num_ants": SOURCE_NUM_ANTS,
    "ant_stages": ANT_STAGES,
    "updates_per_stage": GLOBAL_UPDATE_CAP,
}


## Train Ant Stages


In [ ]:
ant_count_result = workflows.run_ant_count_curriculum(
    ant_stages=ANT_STAGES,
    source_checkpoint=SOURCE_COMMUNICATION_CHECKPOINT,
    source_num_ants=SOURCE_NUM_ANTS,
    communication_bits=COMMUNICATION_BITS,
    run_dir=RUN_DIR,
    common_args=COMMON_ARGS,
    experiment_name=experiment.args.get("exp_name", experiment.name),
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
)
FINAL_ANT_COUNT_CHECKPOINT = ant_count_result["final_checkpoint"]
ant_count_result


## Optional Render and Vault


In [ ]:
rollout_result = workflows.render_ant_count_rollouts(
    experiment_config=EXPERIMENT_CONFIG,
    source_checkpoint=SOURCE_COMMUNICATION_CHECKPOINT,
    run_dir=RUN_DIR,
    media_dir=MEDIA_DIR,
    communication_bits=COMMUNICATION_BITS,
    source_num_ants=SOURCE_NUM_ANTS,
    ant_stages=ANT_STAGES,
    global_update_cap=GLOBAL_UPDATE_CAP,
    tile_size=ROLLOUT_TILE_SIZE,
)
rollout_result
